# 01 — Czyszczenie danych

## Cel

Celem projektu jest przewidzenie, czy firma zbankrutuje w kolejnym roku na podstawie wskaźników finansowych.

- `class = 0` — brak bankructwa,
- `class = 1` — bankructwo.

W tym notebooku wczytuję dane, sprawdzam podstawowe problemy i zapisuję jeden oczyszczony plik CSV.

Nie uzupełniam jeszcze braków medianą i nie skaluję danych. Zrobię to później w `Pipeline`, żeby przygotowanie danych nie korzystało z informacji ze zbioru testowego.

## 1. Importy i wczytanie danych

In [8]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import arff
from sklearn.model_selection import train_test_split

# Wszystkie pliki z danymi trzymam w jednym katalogu.
DATA_DIR = Path("dane")
RAW_PATH = DATA_DIR / "5year.arff"
CLEAN_PATH = DATA_DIR / "5year_cleaned.csv"

# ARFF wczytuję przez scipy, a potem zamieniam na DataFrame.
raw_data, metadata = arff.loadarff(RAW_PATH)
df = pd.DataFrame(raw_data)

# W zbiorze jest 64 wskaźniki finansowe Attr1-Attr64.
feature_cols = [f"Attr{i}" for i in range(1, 65)]

# Target w pliku ARFF jest zapisany jako bytes: b"0" i b"1".
# Zamieniam go na zwykłe liczby 0/1.
df["class"] = df["class"].map({b"0": 0, b"1": 1}).astype(int)

print("Wymiary surowego zbioru:", df.shape)
print("Liczba cech:", len(feature_cols))
display(df.head())

Wymiary surowego zbioru: (5910, 65)
Liczba cech: 64


,Attr1,Attr2,Attr3,Attr4,Attr5,Attr6,Attr7,Attr8,Attr9,Attr10,...,Attr56,Attr57,Attr58,Attr59,Attr60,Attr61,Attr62,Attr63,Attr64,class
0,0.088238,0.55472,0.01134,1.0205,-66.5200,0.342040,0.109490,0.57752,1.0881,0.32036,...,0.080955,0.275430,0.91905,0.002024,7.2711,4.7343,142.760,2.5568,3.2597,0
1,-0.006202,0.48465,0.23298,1.5998,6.1825,0.000000,-0.006202,1.06340,1.2757,0.51535,...,-0.028591,-0.012035,1.00470,0.152220,6.0911,3.2749,111.140,3.2841,3.3700,0
2,0.130240,0.22142,0.57751,3.6082,120.0400,0.187640,0.162120,3.05900,1.1415,0.67731,...,0.123960,0.192290,0.87604,0.000000,8.7934,2.9870,71.531,5.1027,5.6188,0
3,-0.089951,0.88700,0.26927,1.5222,-55.9920,-0.073957,-0.089951,0.12740,1.2754,0.11300,...,0.418840,-0.796020,0.59074,2.878700,7.6524,3.3302,147.560,2.4735,5.9299,0
4,0.048179,0.55041,0.10765,1.2437,-22.9590,0.000000,0.059280,0.81682,1.5150,0.44959,...,0.240400,0.107160,0.77048,0.139380,10.1180,4.0950,106.430,3.4294,3.3622,0


### Wniosek

Surowy zbiór ma **5910 obserwacji**, **64 cechy finansowe** i jedną zmienną docelową `class`.

Wszystkie cechy wejściowe są liczbowe, więc nie będzie potrzebne kodowanie zmiennych kategorycznych.

## 2. Duplikaty, wartości nieskończone i braki danych

In [9]:
# Najpierw sprawdzam, czy występują wartości inf / -inf.
# Gdyby występowały, zamieniam je na NaN i później obsłuży je imputer.
inf_count = int(np.isinf(df[feature_cols].to_numpy()).sum())
df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan)

# Pełny duplikat oznacza identyczny cały wiersz, razem z targetem.
duplicate_count = int(df.duplicated().sum())

# Zostawiam tylko pierwszą kopię powtarzającego się rekordu.
df = df.drop_duplicates().reset_index(drop=True)

print("Wartości inf / -inf:", inf_count)
print("Usunięte duplikaty:", duplicate_count)
print("Wymiary po czyszczeniu:", df.shape)
print("Łączna liczba braków:", int(df[feature_cols].isna().sum().sum()))

Wartości inf / -inf: 0
Usunięte duplikaty: 60
Wymiary po czyszczeniu: (5850, 65)
Łączna liczba braków: 4634


### Wniosek

Nie znaleziono wartości `inf` ani `-inf`.

Usunięto **60 pełnych duplikatów**, dlatego liczba obserwacji spadła z 5910 do **5850**.

W danych pozostały **4634 braki**. Nie usuwam z ich powodu całych wierszy, ponieważ straciłbym sporo danych. Braki zostaną uzupełnione medianą dopiero wewnątrz `Pipeline`.

## 3. Krótka kontrola wartości skrajnych

In [10]:
# q01 i q99 pokazują granice środkowych 98% obserwacji.
# Porównuję je z minimum, maksimum i medianą, aby zobaczyć skalę wartości skrajnych.
q01 = df[feature_cols].quantile(0.01)
q99 = df[feature_cols].quantile(0.99)

extreme_summary = pd.DataFrame({
    "min": df[feature_cols].min(),
    "q01": q01,
    "median": df[feature_cols].median(),
    "q99": q99,
    "max": df[feature_cols].max(),
})

# Pokazuję tylko kilka przykładów, zamiast drukować 64 wiersze.
display(extreme_summary.loc[["Attr5", "Attr15", "Attr27", "Attr60"]].round(2))

,min,q01,median,q99,max
Attr5,-1076400.00,-719.73,0.49,694.66,1250100.0
Attr15,-9632400.00,-21404.36,876.08,38137.87,1341700.0
Attr27,-158130.00,-99.16,0.98,5797.12,565940.0
Attr60,-12.44,1.30,9.04,693.12,4818700.0


### Wniosek

Wartości skrajne są bardzo duże w porównaniu z medianami. Przykładowo `Attr60` ma medianę około **9**, a maksimum przekracza **4,8 mln**.

Nie traktuję takich wartości automatycznie jako błędów. Przy wskaźnikach finansowych skrajny wynik może opisywać rzeczywistą, nietypową sytuację firmy. Dlatego zostawiam je w danych i dokładniej oglądam rozkłady w EDA.

## 4. Rozkład klas

In [11]:
# Liczę zarówno liczbę obserwacji, jak i procent każdej klasy.
target_summary = pd.DataFrame({
    "count": df["class"].value_counts().sort_index(),
    "percent": (
        df["class"]
        .value_counts(normalize=True)
        .sort_index() * 100
    ).round(2),
})

display(target_summary)

,count,percent
class,,
0,5442,93.03
1,408,6.97


### Wniosek

Klasa `1` stanowi tylko **6,97%** danych, więc zbiór jest mocno niezbalansowany.

Sama `accuracy` może przez to być myląca. W dalszej części będę sprawdzał także `precision`, `recall`, `F1`, `ROC-AUC` i **Average Precision (AP)**.

## 5. Podział używany w dalszych notebookach

In [12]:
# Najpierw odkładam 20% danych na końcowy test.
development, test = train_test_split(
    df,
    test_size=0.20,
    stratify=df["class"],  # zachowuję podobny udział klas
    random_state=42
)

# Z pozostałych 80% odkładam 20% na walidację.
# Daje to około 64% / 16% / 20% całego zbioru.
train, validation = train_test_split(
    development,
    test_size=0.20,
    stratify=development["class"],
    random_state=42
)

print("Trening:", len(train))
print("Walidacja:", len(validation))
print("Test:", len(test))

Trening: 3744
Walidacja: 936
Test: 1170


### Dlaczego taki podział?

Powstają trzy części:

- **3744** obserwacje treningowe,
- **936** walidacyjnych,
- **1170** testowych.

EDA będzie wykonywana tylko na treningu. Walidację wykorzystam później do wyboru progu, a test pozostanie odłożony aż do końcowej oceny modelu.

## 6. Zapis oczyszczonych danych

In [13]:
# Zapisuję jeden oczyszczony plik.
# Kolejne notebooki odtwarzają z niego zawsze ten sam podział dzięki random_state=42.
df.to_csv(CLEAN_PATH, index=False)

print("Zapisano:", CLEAN_PATH)
print("Wymiary:", df.shape)

Zapisano: dane/5year_cleaned.csv
Wymiary: (5850, 65)


## Podsumowanie

Po czyszczeniu pozostaje 5850 obserwacji i 64 cechy.

Najważniejsze decyzje:

- usunąłem pełne duplikaty,
- wartości `inf` zamieniłem na braki,
- braków nie uzupełniałem przed podziałem danych,
- wartości skrajne zostawiłem,
- zapisałem jeden plik `dane/5year_cleaned.csv`.

Następny krok: `02_eda.ipynb`.